# HW2 – 4. One-stage anchor-free detector (YOLO11)

YOLO11s (Ultralytics, COCO-pretrained) trained under the protocol from notebook 01 with the same budget as Faster R-CNN (notebook 02) and RetinaNet (notebook 03). YOLO11 is anchor-free (one box per feature-map location, predicted as distances to the four box sides), assigns locations to objects with the task-aligned assigner (TAL) and still uses NMS. YOLO26 is not used: it is NMS-free by default, which would mix two changes in the comparison with the anchor-based models.

**Experiments** (each starts from the best configuration so far):
1. **High-resolution detection head** – P3–P5 (strides 8–32, default) vs an extra P2 level (stride 4).
2. **Box regression loss** – CIoU (default) vs a 50/50 mix of CIoU and the Normalized Wasserstein Distance (NWD) loss for tiny boxes.
3. **Test-time input resolution × NMS IoU** on the best training (no retraining).

We explain how locations are assigned to objects without anchors and how this affects small and dense objects, compare with the tuned anchor-based detectors by object size and analyse failure cases.

## 0. Setup

In [ ]:
# Make the repo importable: Kaggle clones it, locally the root is the parent of notebooks/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub ultralytics==8.4.207   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [ ]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import ultralytics
from torchvision.ops import box_iou
from utils.common import *
from utils.ultra import ULTRA_VERSION, YOLO_DATA_DIR, export_yolo, load_ultra, nwd, train_runs, ultra_predictor

seed_everything()
print(env_info(), "| ultralytics:", ultralytics.__version__, "| data:", RAW_DIR)
if ultralytics.__version__ != ULTRA_VERSION:
    print(f"Warning: tested with ultralytics {ULTRA_VERSION}; the loss/assigner patches in utils/ultra.py depend on its internals")

Stages run in this session are listed in `RUN`; finished stages are downloaded from the Hugging Face repo. With 2 GPUs the baseline and the P2 run of experiment 1 are trained concurrently (~3 h for all stages on 2×T4, ~4.3 h on one T4). `QUICK=True` is a local smoke test on a few frames.

In [ ]:
RUN = {"baseline", "p2", "loss", "final"}  # stages computed in this session; the rest are pulled from the HF repo
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed)

# Fail early if trained stages could not be uploaded (needs HF_TOKEN with write access)
if RUN and not QUICK:
    check_hf_write()

## 1. Training setup

- **Model:** YOLO11s (9.5M parameters; 9.6M with the P2 level), COCO weights, new 8-class head. Box branch: distribution focal loss (DFL) over the distances to the four sides + CIoU loss; classification: BCE against the TAL soft targets (section 3).
- **Data:** the same frames as notebooks 02–03 (every 3rd, 8,090 train / 951 val), resized once to a 1280 px long side (`export_yolo`) so that JPEG decoding does not bottleneck the 4 Kaggle CPUs. Labels: the provided YOLO txt files (`iscrowd=0` boxes only, ignore regions are not targets). Augmentation: Ultralytics' standard recipe (mosaic, flip 0.5, HSV, translation) with scale jitter ±25% as in `PREP`; mosaic is switched off for the last 2 epochs.
- **Optimisation:** AdamW, lr 0.001 (Ultralytics' own choice for runs under 10k iterations), nominal batch 64 (batch 8 × 8 accumulation steps), 1 warm-up epoch, then linear decay to 1% (Ultralytics default), AMP, seed 42.
- **Budget:** 1.1 h of training per run (incl. validation) = the 67 min of the Faster R-CNN baseline (3 epochs on a T4, `runs/frcnn_baseline/history.json`). Ultralytics' `time` argument re-estimates the number of epochs after each epoch, so the LR schedule ends with the budget. With 2 GPUs, two independent runs are trained concurrently, one per GPU; they share the CPUs, so the epochs reached are reported per run.
- **Evaluation:** the shared `coco_evaluate` on the val frames (ignore regions as `iscrowd`, up to 300 detections, score ≥ 0.001), on Ultralytics' best epoch (chosen by its own val fitness, which does not exclude ignore regions); input 1280 px, NMS IoU 0.7 (Ultralytics default); speed at batch size 1, FP16.

In [ ]:
CFG = dict(
    arch="yolo11s", weights="yolo11s.pt", long_side=640 if QUICK else PREP["long_side"],
    train_frames=16 if QUICK else None, val_frames=8 if QUICK else None, num_workers=4, seed=SEED,
    train=dict(  # Ultralytics train() arguments shared by every run
        imgsz=640 if QUICK else PREP["long_side"], batch=2 if QUICK else 8, epochs=1 if QUICK else 100,
        time=None if QUICK else 1.1, optimizer="AdamW", lr0=0.001, warmup_epochs=0 if QUICK else 1,
        close_mosaic=0 if QUICK else 2, scale=0.25, workers=2 if QUICK else 4, seed=SEED, deterministic=True,
        max_det=EVAL_DETS, plots=False, cache=False),
)
DEFAULT_MODEL = dict(p2=False, nwd_ratio=0.0, nwd_c=32.0)   # Ultralytics defaults (P3-P5, no NWD)
NWD = dict(nwd_ratio=0.5, nwd_c=32.0)        # box loss = 0.5 CIoU + 0.5 NWD; C = median box size at 1280 px (notebook 01)
DEFAULT_TEST = dict(imgsz=CFG["long_side"], nms=0.7)
GRID = dict(imgsz=[640, 800] if QUICK else [1280, 1536, 1920], nms=[0.6, 0.7])   # test-time grid on the best training


def name(stage):
    return f"yolo_{stage}" + ("_quick" if QUICK else "")


print(CFG)

## 2. Data export and helpers

Ultralytics reads images and YOLO labels from disk. The kept frames are resized once to the training resolution and their label files copied next to them (`utils/ultra.py`); the frames, classes and resolution are the same as for `DetDataset`. On Kaggle the copy goes to `/tmp` (about 2.5 GB), not to the persisted `/kaggle/working`.

In [ ]:
TRAIN_STAGES = {"baseline", "p2", "loss"}
DATA_YAML = None
if RUN & TRAIN_STAGES:
    DATA_YAML = export_yolo(YOLO_DATA_DIR / ("quick" if QUICK else str(CFG["long_side"])), long_side=CFG["long_side"],
                            max_frames=CFG["train_frames"])

In [ ]:
def ensure_run(stage, weights=False):
    # Local folder of a finished stage; downloaded from the HF repo if it was produced in another session
    return fetch_run(name(stage), weights, hint=f"Add '{stage}' to RUN to compute it in this session (RUN = {RUN}).")


def model_cfg(stage):
    return json.load(open(ensure_run(stage) / "config.json"))["model"]


def load_model(stage):
    return load_ultra(ensure_run(stage, weights=True) / "best.pt")


def evaluate(model, imgsz=DEFAULT_TEST["imgsz"], nms=DEFAULT_TEST["nms"]):
    return evaluate_model(ultra_predictor(model, imgsz, nms), "val", max_frames=CFG["val_frames"], batch_size=1,
                          num_workers=CFG["num_workers"], label_offset=0, long_side=imgsz)


def best_so_far(stages):
    # Finished stage with the highest val mAP (optional stages that were never run are skipped)
    done = {}
    for s in stages:
        try:
            done[s] = json.load(open(ensure_run(s) / "metrics.json"))["mAP"]
        except FileNotFoundError:
            pass
    return max(done, key=done.get)

In [ ]:
def train_stages(stages):
    # Train the given stages ({stage: model settings}) under the fixed protocol, concurrently on 2 GPUs if available,
    # then evaluate each best epoch with the shared COCO evaluation, save and upload the run
    cfgs = {s: dict(name=name(s), data=str(DATA_YAML), arch=CFG["arch"], weights=CFG["weights"],
                    **{**DEFAULT_MODEL, **m}, train=CFG["train"]) for s, m in stages.items()}
    train_runs(list(cfgs.values()))
    for s, c in cfgs.items():
        model = load_ultra(run_dir(c["name"]) / "best.pt")
        metrics = evaluate(model)
        save_run(c["name"], config={**CFG, "model": {k: c[k] for k in DEFAULT_MODEL}, "test": DEFAULT_TEST},
                 metrics=metrics)                               # history.json is written by the training process
        if not QUICK:
            push_run(c["name"])
        del model
        torch.cuda.empty_cache()


def show(stages):
    # Metrics of finished stages, one row per stage
    return pd.DataFrame({s: {k: v for k, v in json.load(open(ensure_run(s) / "metrics.json")).items() if k != "per_class"}
                         for s in stages}).T.round(3)

## 3. How locations are assigned to objects without anchors

YOLO11 predicts one box per feature-map location: the point at the centre of each grid cell, $((i + 0.5)\,s, (j + 0.5)\,s)$ for stride $s$ (8, 16, 32; plus 4 with P2), regresses the distances to the four box sides and scores the 8 classes. There are no anchor shapes, so which locations learn an object is decided by the **task-aligned assigner** (TAL, from TOOD), recomputed for every batch:

1. **Candidates:** the points that lie inside the GT box, on all levels. Boxes with a side shorter than the second-smallest stride (16 px; 8 px with P2) are first enlarged to that size around their centre, so that tiny boxes still contain points.
2. **Alignment metric** of each candidate: $t = \text{score}^{0.5} \cdot \text{CIoU}^{6}$, from the model's current class score and predicted box.
3. **Top-k** (10) candidates by $t$ become positives; a point selected by several GTs goes to the GT with the highest CIoU.
4. **Soft targets:** the classification target of a positive is $t$ normalised to the best CIoU of its GT, so the score also learns localisation quality.

Compared with the anchor IoU thresholds of RetinaNet (notebook 03), no shape prior is needed and the positives follow the predictions as training goes on. **Small objects** contain few grid points, so they have few candidates, and part of them come from the enlarged box, i.e. from locations whose centre lies off the object. **Dense objects** compete for the same points: a point inside several boxes can be positive for only one of them.

Measured below on 300 train frames at the input resolution, without training: candidate points per GT box, the share that lies on the object itself, GT boxes without any candidate on the object, and GT boxes whose candidates are all shared with another box, for P3–P5 and P2–P5.

In [ ]:
def _points(lo, hi, s, n):
    # Index range of the grid points (i + 0.5) * s strictly inside (lo, hi), clipped to the n points of the level
    first = np.clip(np.floor(lo / s - 0.5).astype(int) + 1, 0, n)
    last = np.clip(np.ceil(hi / s - 0.5).astype(int) - 1, -1, n - 1)
    return first, last, np.maximum(last - first + 1, 0)


def candidate_stats(strides, n_frames=10 if QUICK else 300, long_side=CFG["long_side"]):
    # Per GT box: TAL candidate points over the given levels, those on the object (inside the original box) and
    # those shared with another GT of the same frame; boxes at the input resolution, train frames
    frames, boxes, _ = load_split("train")
    frames = subsample(frames).sample(n_frames, random_state=SEED)
    by_img = dict(tuple(boxes[boxes.image_id.isin(frames.image_id)].groupby("image_id")))
    min_side = strides[1]                                   # Ultralytics enlarges sides below the second-smallest stride
    rows = []
    for f in frames.itertuples():
        b = by_img.get(f.image_id)
        if b is None:
            continue
        k = long_side / max(f.width, f.height)
        x1, y1, w, h = (b[c].to_numpy() * k for c in ["x", "y", "w", "h"])
        cx, cy = x1 + w / 2, y1 + h / 2
        ew, eh = np.maximum(w, min_side), np.maximum(h, min_side)
        cand, on, shared = np.zeros(len(b)), np.zeros(len(b)), np.zeros(len(b))
        for s in strides:
            nx, ny = int(np.ceil(f.width * k / s)), int(np.ceil(f.height * k / s))
            ax0, ax1, anx = _points(cx - ew / 2, cx + ew / 2, s, nx)
            ay0, ay1, any_ = _points(cy - eh / 2, cy + eh / 2, s, ny)
            cand += anx * any_
            on += _points(x1, x1 + w, s, nx)[2] * _points(y1, y1 + h, s, ny)[2]
            # number of GT candidate rectangles covering each point (2D difference array), then shared points per GT
            ok = (anx > 0) & (any_ > 0)
            cnt = np.zeros((ny + 1, nx + 1))
            for dy, dx, v in [(ay0, ax0, 1), (ay0, ax1 + 1, -1), (ay1 + 1, ax0, -1), (ay1 + 1, ax1 + 1, 1)]:
                np.add.at(cnt, (dy[ok], dx[ok]), v)
            multi = np.pad((cnt.cumsum(0).cumsum(1) >= 2)[:ny, :nx], ((1, 0), (1, 0))).cumsum(0).cumsum(1)
            sh = multi[ay1 + 1, ax1 + 1] - multi[ay0, ax1 + 1] - multi[ay1 + 1, ax0] + multi[ay0, ax0]
            shared += np.where(ok, sh, 0)
        rows.append(pd.DataFrame({"size": np.sqrt(w * h), "cand": cand, "on": on, "shared": shared}))
    df = pd.concat(rows)
    df["bucket"] = pd.cut(df["size"], [0, 16, 32, 96, 1e9], labels=["tiny (<16)", "small (16-32)", "medium", "large"])
    g = df.groupby("bucket", observed=False)
    return pd.DataFrame({
        "GT boxes, %": g.size() / len(df) * 100,
        "candidate points / GT": g.cand.mean(),
        "% candidates on the object": g.on.sum() / g.cand.sum() * 100,
        "% GT with no candidate on the object": g.on.apply(lambda v: (v == 0).mean() * 100),
        "% GT with < 10 candidates (top-k)": g.cand.apply(lambda v: (v < 10).mean() * 100),
        "% GT with all candidates shared": g.apply(lambda d: ((d.shared >= d.cand) & (d.cand > 0)).mean() * 100,
                                                   include_groups=False),
    })


assign = pd.concat({"P3-P5 (strides 8, 16, 32)": candidate_stats((8, 16, 32)),
                    "P2-P5 (strides 4, 8, 16, 32)": candidate_stats((4, 8, 16, 32))})
assign.round(1)

**Findings.** On P3–P5, tiny boxes (< 16 px, 11% of the boxes) get only about 6 candidate points, 41% of which lie off the object (they exist only because the box is enlarged to 16 px), and 99% of them have fewer candidates than the top-k of 10, so every candidate becomes a positive, good or bad. 10% of the tiny boxes have all their candidates shared with a neighbour, so in dense scenes they compete with other objects for every location. Small boxes (16–32 px, 44%) fare better (12 candidates, 97% on the object), and medium and large boxes have far more candidates than top-k, so for them TAL picks the best-aligned few. The P2 level multiplies the candidates of tiny boxes by 2.4 (14 per box, 97% on the object) and halves the share of fully shared ones (3.6%), so it directly addresses the weakest group.

## 4. Baseline

Ultralytics defaults: P3–P5 levels, CIoU + DFL box loss, TAL top-k 10, NMS IoU 0.7. The baseline and the P2 run of experiment 1 do not depend on each other, so they are trained together here (concurrently when 2 GPUs are available).

In [ ]:
train_stages({s: m for s, m in {"baseline": {}, "p2": {"p2": True}}.items() if s in RUN})
show(["baseline"])

## 5. Experiment 1 – High-resolution detection head (P2)

54% of the boxes are under 32 px at the input size and 10% under 16 px (notebook 01), the region proposals of Faster R-CNN cover only 71% of the small objects (notebook 02), and section 3 shows that on P3–P5 most candidate points of tiny boxes lie off the object. The P2 variant (`utils/models/yolo11-p2.yaml`, following Ultralytics' `yolov8-p2.yaml`) adds a stride-4 level fed by the stride-4 backbone stage; the backbone and the top-down path keep their COCO weights, the P2 branch and the bottom-up path are new. Cost: the stride-4 map has 4× the locations of P3, so training and inference are slower and fewer epochs fit the budget.

In [ ]:
show(["baseline", "p2"])

_TODO: findings after the run._

## 6. Experiment 2 – Box regression loss (NWD)

Loss modification on the best configuration of experiment 1. IoU-based losses are very sensitive to small shifts of tiny boxes (table below): a 2 px shift in x and y drops the IoU of an 8 px box to 0.39, while a 96 px box keeps 0.92, and boxes that do not overlap get no IoU gradient. The **Normalized Wasserstein Distance** (Wang et al., 2021) models each box as a 2D Gaussian and compares them with the Wasserstein distance, $\text{NWD} = \exp(-W_2 / C)$: it depends on the absolute pixel offset, not on the box size, and stays informative without overlap. $C = 32$ px is the median box size at 1280 px (notebook 01). The box loss becomes $0.5\,(1 - \text{CIoU}) + 0.5\,(1 - \text{NWD})$, keeping CIoU's scale invariance for larger boxes; the assigner still uses CIoU.

In [ ]:
# IoU vs NWD of a square box and the same box shifted by 2 px (x and y), for several sizes
sizes = [8, 16, 32, 96]
a = torch.tensor([[0, 0, s, s] for s in sizes], dtype=torch.float32)
b = a + 2
pd.DataFrame({"IoU": box_iou(a, b).diag().numpy(), f"NWD (C = {NWD['nwd_c']:.0f})": nwd(a, b, NWD["nwd_c"]).numpy()},
             index=pd.Index(sizes, name="box size, px (shift 2 px in x and y)")).round(2)

In [ ]:
if "loss" in RUN:
    train_stages({"loss": {**model_cfg(best_so_far(["baseline", "p2"])), **NWD}})
show(["baseline", "p2", "loss"])

_TODO: findings after the run._

## 7. Tuned model: test-time input resolution × NMS IoU

Final model ("tuned"): the best training by val mAP, evaluated on a grid of input resolutions (long side 1280 / 1536 / 1920 px) and final NMS IoU (0.6 / 0.7), both at test time. A higher input resolution enlarges small objects (notebook 01: 10% of boxes are under 16 px at 1280 px, 2% at 1920 px); the val set has 34% 4K frames and the test frames are up to 1920 px wide. Settings are selected on val; its predictions are saved for the analysis below.

In [ ]:
TRAINED = ["baseline", "p2", "loss"]
if "final" in RUN:
    src = best_so_far(TRAINED)
    model = load_model(src)
    rows, best = [], None
    for sz in GRID["imgsz"]:
        ds = DetDataset("val", label_offset=0, max_frames=CFG["val_frames"], long_side=sz)
        for t in GRID["nms"]:
            p, ms = predict_dataset(ultra_predictor(model, sz, t), ds, batch_size=1, num_workers=CFG["num_workers"])
            m = {**coco_evaluate(p, "val", image_ids=ds.image_ids), "ms_per_img": ms}
            rows.append({"imgsz": sz, "nms": t, **{k: v for k, v in m.items() if k != "per_class"}})
            if best is None or m["mAP"] > best[1]["mAP"]:
                best = (dict(imgsz=sz, nms=t), m, p)
    test, metrics, p = best
    d = save_run(name("final"), config={"trained": name(src), "model": model_cfg(src), "test": test, "image_ids": ds.image_ids},
                 metrics=metrics, preds=p)
    pd.DataFrame(rows).to_csv(d / "grid.csv", index=False)
    if not QUICK:
        push_run(name("final"))
    del model
    torch.cuda.empty_cache()
d = ensure_run("final")
final_cfg, final = json.load(open(d / "config.json")), json.load(open(d / "metrics.json"))
preds = json.load(open(d / "preds_val.json"))
grid = pd.read_csv(d / "grid.csv")
print("trained:", final_cfg["trained"], "| best test-time setting:", final_cfg["test"])
pd.concat({m: grid.pivot(index="imgsz", columns="nms", values=m) for m in ["mAP", "AP_small", "ms_per_img"]}, axis=1).round(3)

In [ ]:
# Search range and selected setting of every tuned parameter (each selected by val mAP; see the tables above)
sel = final_cfg["model"]
loss_name = lambda m: f"{1 - m['nwd_ratio']:g} CIoU + {m['nwd_ratio']:g} NWD (C = {m['nwd_c']:g})" if m["nwd_ratio"] else "CIoU"
pd.DataFrame({
    "search range": {"detection levels (training)": "P3-P5, P2-P5", "box loss (training)": f"CIoU, {loss_name(NWD)}",
                     "input long side, px (test)": ", ".join(map(str, GRID["imgsz"])), "NMS IoU (test)": ", ".join(map(str, GRID["nms"]))},
    "selected": {"detection levels (training)": "P2-P5" if sel["p2"] else "P3-P5", "box loss (training)": loss_name(sel),
                 "input long side, px (test)": str(final_cfg["test"]["imgsz"]), "NMS IoU (test)": str(final_cfg["test"]["nms"])},
}).rename_axis("parameter")

_TODO: findings after the run._

## 8. Comparison with the anchor-based detectors

Same val frames and evaluation for all models. Rows: tuned Faster R-CNN (notebook 02), tuned RetinaNet (notebook 03, if it has been run), the YOLO trainings (1280 px, NMS 0.7) and the tuned YOLO model. Results by object size: COCO AP_small / AP_medium / AP_large, and the share of GT boxes covered by a detection (IoU ≥ 0.5, any class, score ≥ 0.3).

In [ ]:
# Tuned anchor-based models from notebooks 02-03 (metrics, predictions and training history, no weights)
ANCHOR_BASED = {"frcnn_final": "Faster R-CNN, tuned (notebook 02)", "retina_final": "RetinaNet, tuned (notebook 03)"}
others = {}
for r, label in ANCHOR_BASED.items():
    try:
        fetch_run(r)
        others[r] = label
    except FileNotFoundError:
        print(f"{label}: no results yet ({r}), left out of the comparison")


def row_name(stage):
    # levels and box loss used in training + test-time input size and NMS IoU
    c = json.load(open(ensure_run(stage) / "config.json"))
    mc, t = c["model"], c["test"]
    parts = ["P2-P5" if mc["p2"] else "P3-P5", f"CIoU+NWD {mc['nwd_ratio']}" if mc["nwd_ratio"] else "CIoU"]
    return f"{'final: ' if stage == 'final' else ''}YOLO11s {', '.join(parts)}, {t['imgsz']} px, NMS {t['nms']}"


stages = [s for s in TRAINED + ["final"] if (RUNS_DIR / name(s) / "metrics.json").exists()]
FINAL_NAME = row_name("final")
cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR", "ms_per_img"]
table = load_results().reindex(list(others) + [name(s) for s in stages])[cols]
table.index = list(others.values()) + [row_name(s) for s in stages]
table

In [ ]:
# Share of GT boxes found by size and detections per image (score >= 0.3), and per-class AP
gt, det = match_errors(preds, final_cfg["image_ids"])
found, per_class = {"YOLO11s, tuned": size_coverage(gt)}, {"YOLO11s, tuned": final["per_class"]}
n_det = {"YOLO11s, tuned": len(det) / len(final_cfg["image_ids"])}
for r, label in others.items():
    c = json.load(open(RUNS_DIR / r / "config.json"))
    g, dt = match_errors(json.load(open(RUNS_DIR / r / "preds_val.json")), c["image_ids"])
    found[label], n_det[label] = size_coverage(g), len(dt) / len(c["image_ids"])
    per_class[label] = json.load(open(RUNS_DIR / r / "metrics.json"))["per_class"]
cmp = pd.DataFrame(found).rename(index=lambda b: f"GT found, {b}")
cmp.loc["detections / image"] = n_det
display(cmp.round(3), pd.DataFrame(per_class).T.round(3).style.set_caption("AP per class"))

Training curves of the YOLO runs. The validation mAP here is Ultralytics' own (computed during training; detections in ignore regions count as false positives), so it is lower than the `coco_evaluate` numbers above and is not comparable with the curves of notebooks 02–03; the right plot shows how many epochs each run fit into the budget.

In [ ]:
plot_history({row_name(s): json.load(open(ensure_run(s) / "history.json")) for s in stages if s != "final"})

_TODO: findings after the run._

## 9. Failure cases

Errors of the tuned model at score ≥ 0.3 and IoU 0.5: outcome of every GT box by size and occlusion, class confusions and the types of false positives (same definitions as in notebook 02).

GT outcomes: **correct** = detected with the right class; **crowded** = covered only by a box already used for a neighbour; **wrong_class** = detected as another class; **mislocalized** = best IoU 0.1–0.5; **missed** = no detection. Detections: **tp** = matches a GT box; **background** = on nothing; **ignored** = centre in an ignore region; **wrong_class** = right place, wrong class; **duplicate** = second box on an already matched GT.

In [ ]:
error_tables(gt, det)
plot_confusion(gt);

Frames with the most missed small objects, the most class confusions and the most background false positives. Top: ground truth (box colour = class). Bottom: model predictions with score ≥ 0.3 (box colour = predicted class). Red areas are ignore regions.

In [ ]:
plot_failure_frames(gt, det, preds);

_TODO: failure cases and their likely causes after the run._

## 10. Conclusion

_TODO: after the run._